# neural_chess_engine -- run9 Colab launcher (arms a / b / c)

Thin launcher only: picks the arm's branch, installs deps, copies the pre-encoded v2 shards from Drive to local disk, runs `src/train.py` in this kernel (live `tqdm`), then scores the result on the held-out test month. No training logic lives here.

| arm | branch | what changes vs run6 |
|---|---|---|
| a | `run9`  | per-layer GAB bias (control) |
| b | `run9b` | a + pairwise (bilinear) move head + legal-masked joint loss |
| c | `run9c` | a + attack/defence-graph attention bias, SEE channel off |

**One arm per Colab session.** Set `ARM` in the config cell, then run top to bottom. Pick a GPU runtime first (`Runtime > Change runtime type`). The v2 shards (`tensors/v2/<month>/{train,val}` + `tensors/v2/test_2025-04/test`) must already be on Drive -- they are encoded locally, not here. Checkpoints go straight to `checkpoints/run9{arm}/` on the mounted Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("!! no GPU -- go to Runtime > Change runtime type > GPU before continuing")

In [ ]:
ARM = "a"  # <-- "a" (control), "b" (pairwise head + legal loss) or "c" (attack-graph bias)

ARMS = {
    "a": ("run9",  "--gab-per-layer"),
    "b": ("run9b", "--gab-per-layer --pair-head --legal-loss"),
    "c": ("run9c", "--gab-per-layer --attack-bias --no-see-channel"),
}
GIT_BRANCH, ARM_FLAGS = ARMS[ARM]
RUN_NAME = f"run9{ARM}"

REPO_DIR = "/content/chess_bot"
REPO_URL = "https://github.com/mericberktas/neural_chess_engine.git"
DRIVE_BASE = "/content/drive/MyDrive/chess_bot"
V2_DIR = f"{DRIVE_BASE}/tensors/v2"
CKPT_DIR = f"{DRIVE_BASE}/checkpoints/{RUN_NAME}"

# the same 16 train months run6 used; 2025-04 stays the held-out test month
MONTHS = [
    "2026-08", "2026-07", "2026-06", "2026-05", "2026-04", "2026-03", "2026-02", "2026-01",
    "2025-12", "2025-11", "2025-10", "2025-09", "2025-08", "2025-07", "2025-06", "2025-05",
]
TEST_MONTH = "2025-04"

BATCH_SIZE = 256
LR = 3e-4
VAL_INTERVAL = 2000
EPOCHS = 15
PATIENCE = 4
print(f"arm {ARM}: branch {GIT_BRANCH}, run {RUN_NAME}, flags: {ARM_FLAGS}")

In [ ]:
import os

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !cd {REPO_DIR} && git fetch origin && git checkout {GIT_BRANCH} && git pull origin {GIT_BRANCH}
elif os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already exists (manually uploaded) -- using it as-is, not cloning.")
else:
    !git clone --branch {GIT_BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

In [ ]:
# Colab already ships a CUDA-enabled torch -- skip it like runpod_overnight.sh does,
# installing our own could mismatch the preinstalled CUDA build.
!grep -v '^torch$' requirements.txt > /tmp/requirements-no-torch.txt
!pip install --quiet -r /tmp/requirements-no-torch.txt

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, f"{REPO_DIR}/src")

TRAIN_DIRS = [f"{V2_DIR}/{m}/train" for m in MONTHS]
VAL_DIRS = [f"{V2_DIR}/{m}/val" for m in MONTHS]
TEST_DIR = f"{V2_DIR}/test_{TEST_MONTH}/test"

missing = [d for d in TRAIN_DIRS + VAL_DIRS + [TEST_DIR] if not Path(d).is_dir()]
if missing:
    raise FileNotFoundError(f"{len(missing)} v2 shard dirs missing on Drive, first: {missing[0]} -- upload data/v2 from the local encode first")
print(f"all {len(MONTHS)} train/val month dirs + the test dir found")

In [ ]:
# Drive's FUSE mount is slow on a cold runtime: train.py's ShardDataset opens every shard
# up front just to count positions, and each cold open can take 1-2s -> tens of minutes
# at 0% GPU before the first step. Copy the shards to the VM's local disk in parallel
# first and train from there.
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

LOCAL_DATA = Path("/content/data")

def to_local_path(d):
    return LOCAL_DATA / Path(d).relative_to(V2_DIR)

def copy_shard(src):
    dst = to_local_path(src.parent) / src.name
    if dst.exists():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + ".tmp")  # rename after copy so an interrupted run can't leave a truncated shard behind
    shutil.copy2(src, tmp)
    tmp.replace(dst)

shards = [f for d in TRAIN_DIRS + VAL_DIRS + [TEST_DIR] for f in Path(d).glob("shard_*.npz")]
with ThreadPoolExecutor(32) as pool:
    list(tqdm(pool.map(copy_shard, shards), total=len(shards), desc="copying shards to local disk"))

TRAIN_DIRS = [str(to_local_path(d)) for d in TRAIN_DIRS]
VAL_DIRS = [str(to_local_path(d)) for d in VAL_DIRS]
TEST_DIR = str(to_local_path(TEST_DIR))
print(f"{len(shards)} shards on local disk, training from {LOCAL_DATA}")

In [ ]:
resume_path = f"{CKPT_DIR}/best.pt"
resume_args = []
if os.path.exists(resume_path):
    print(f"found existing checkpoint, resuming from {resume_path}")
    resume_args = ["--resume-from", resume_path]
else:
    print("no existing checkpoint -- starting fresh")

## Train

Runs in this notebook's own kernel via `%run` (not `!python`) so `tqdm`'s bar renders; safe because `train.py` uses `num_workers=0`. Checkpoints (`best.pt`, TensorBoard logs) go straight to the mounted Drive path. Early-stopping ends the run on its own.

In [ ]:
train_dir_args = " ".join(TRAIN_DIRS)
val_dir_args = " ".join(VAL_DIRS)
resume_arg_str = " ".join(resume_args)

cmd = (
    f"src/train.py --train-dir {train_dir_args} --val-dir {val_dir_args} "
    f"--out-dir {CKPT_DIR} "
    f"--batch-size {BATCH_SIZE} --lr {LR} --d-model 256 --nhead 8 --num-layers 6 --dim-feedforward 1024 "
    f"--val-interval {VAL_INTERVAL} --epochs {EPOCHS} --patience {PATIENCE} "
    f"{ARM_FLAGS} {resume_arg_str}"
)
get_ipython().run_line_magic("run", cmd)

## Held-out test month

Scores the best checkpoint on the whole test month (same metric as the val numbers above: legal-masked top-1/3, unmasked in brackets). Run the same cell for every arm -- this is the number to compare.

In [ ]:
!python -u scripts/eval_checkpoint.py --checkpoint {CKPT_DIR}/best.pt --shard-dir {TEST_DIR} --positions 400000